In [ ]:
import pandas as pd

df = pd.read_csv("b2w.csv")[:100].rename(columns={"original_index": "review_id"})[['review_text', 'rating']]
df['resumo'] = 
df['insights'] = ""
df


,review_text,rating,resumo,insights
0,Bem macio e felpudo...recomendo. Preço imbatí...,4,,
1,Produto excepcional! recomendo!!! inovador e ...,5,,
2,recebi o produto antes do prazo mas veio com d...,1,,
3,Bom custo beneficio. Adequado para pessoas que...,5,,
4,Além de higiênico tem o tamanho ideal. Só falt...,3,,
...,...,...,...,...
95,"quanto abri o produto, me dei conta de como as...",2,,
96,gostaria de saber se a parte do recipiente aon...,2,,
97,Foi um ótimo custo x beneficio. Tem uma boa il...,4,,
98,Tudo ocorreu perfeito. Desde o momento do paga...,5,,


In [ ]:
from google import genai
from pydantic import BaseModel
import time
import random

class ReviewAnalysis(BaseModel):
    resumo: str
    insights: str
    analise_sentimento: str | None
    setor_responsavel: str | None
    mensagem_para_cliente: str | None

def analisar_review(review: str):
    client = genai.Client(api_key="")
    PROMPT = f"""
    Você é um analista de e-commerce. Seu objetivo é receber review de clientes e gerar um resumo, insight e análise de sentimento.

    Review: {review}
    Resumo: <no máximo 30 caracteres>
    Insight: <no máximo 100 caracteres> em como podemos melhorar o ecommerce. Se não houver, retorne None.
    Análise de sentimento: <positiva, negativa ou neutra> em relação ao ecommerce.
    Setor responsável: <setor responsável para responder o review (marketing, logística, atendimento ou financeiro)>
    Mensagem para cliente: <mensagem para o cliente para responder o review. O nome da loja é "Rubens Móveis SA">

    retorne em json estruturado com as seguintes chaves:
        
        "resumo": <resumo>,
        "insights": <insights>,
        "analise_sentimento": <análise de sentimento>,
        "setor_responsavel": <setor responsável>,
        "mensagem_para_cliente": <mensagem para o cliente>
        

    """

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=PROMPT.format(review=review),
        config={
            "response_mime_type": "application/json",
            "response_schema": ReviewAnalysis
        }
    )
    print(response.text)
    # Converte a resposta para JSON
    import json
    try:
        response_json = json.loads(response.text)
        return response_json
    except:
        return {
            "resumo": "Erro na análise",
            "insights": "None", 
            "analise_sentimento": "neutra"
        }

count = 0
for index, row in df.iterrows():
    count += 1
    try:
        response = analisar_review(row["review_text"])
        df.loc[index, "resumo"] = response["resumo"]
        df.loc[index, "insights"] = response["insights"] 
        df.loc[index, "analise_sentimento"] = response["analise_sentimento"]
        df.loc[index, "setor_responsavel"] = response["setor_responsavel"]
        df.loc[index, "mensagem_para_cliente"] = response["mensagem_para_cliente"]
        time.sleep(random.uniform(1.5, 4))
    except Exception as e:
        print(f"Erro ao processar linha {index}: {str(e)}")
        df.loc[index, "resumo"] = "Erro na análise"
        df.loc[index, "insights"] = "None"
        df.loc[index, "analise_sentimento"] = "neutra"
        df.loc[index, "setor_responsavel"] = "None"
        df.loc[index, "mensagem_para_cliente"] = "None"
df

{"resumo": "Produto macio, bom preço, entrega rápida", "insights": "None", "analise_sentimento": "positiva", "setor_responsavel": "marketing", "mensagem_para_cliente": "Olá! Ficamos muito felizes em saber que você gostou do produto, do preço e da rapidez da entrega. Agradecemos sua confiança na Rubens Móveis SA!"}
{"resumo": "Produto excepcional, recomendo!", "insights": "None", "analise_sentimento": "positiva", "setor_responsavel": "marketing", "mensagem_para_cliente": "Olá! Ficamos muito felizes que tenha gostado do produto! Agradecemos o seu feedback positivo. Rubens Móveis SA"}
{"resumo": "Produto c/ defeito, não usa", "insights": "Implementar QC rigoroso e acelerar atendimento p/ defeitos, p/ evitar insatisfação c/ a marca.", "analise_sentimento": "negativa", "setor_responsavel": "atendimento", "mensagem_para_cliente": "Olá! Lamentamos o ocorrido. Por favor, entre em contato conosco para que a Rubens Móveis SA possa ajudar a resolver."}
{"resumo": "Bom custo benefício e fácil uso.

KeyboardInterrupt: 

In [25]:
df.to_csv("b2w_analise_completa.csv", index=False)